In [2]:
from math import *
from build123d import *
from ocp_vscode import *
from cad_utils import load_pcb, fast_copy, copy_located, export
from knurled_cylinder import KnurledCylinder

set_port(3939)
set_viewer_config(port=3939)

In [248]:
reset_show()

radius = 10
height = 16
knurl_height = 13
knurls = 16
knurl_depth = 1
wall_thickness = 1.5
outer_wall_thickness = wall_thickness + 1

# shaft is a D shape
shaft_radius = 3
shaft_cut_offset = 1.5
shaft_tolerance = 0.1


with BuildPart() as encoder_knob:
  KnurledCylinder(
    radius, knurl_height, knurls, knurl_depth, align=(Align.CENTER, Align.CENTER, Align.MIN)
  )

  top: Face = faces().sort_by(Axis.Z)[-1]

  with BuildSketch(Plane.XY.offset(height)) as s:
    Circle(radius - (height - knurl_height))

  loft([top, s.sketch])

  with BuildSketch(Plane.XY) as s:
    Circle(radius - outer_wall_thickness)
  extrude(amount=knurl_height, mode=Mode.SUBTRACT)
  with BuildSketch(Plane.XY.offset(knurl_height)) as s:
    Circle(radius - outer_wall_thickness)
  extrude(amount=height - knurl_height - outer_wall_thickness, taper=45, mode=Mode.SUBTRACT)

  with BuildSketch(Plane.XY, mode=Mode.PRIVATE) as shaft:
    Circle(shaft_radius)
    with Locations((0, shaft_cut_offset)):
      Rectangle(
        shaft_radius * 2, shaft_radius * 2, align=(Align.CENTER, Align.MIN), mode=Mode.SUBTRACT
      )

  inner_surface = faces().filter_by(GeomType.CYLINDER).sort_by_distance(Vector())[0]
  inner_bottom_edge: Edge = edges().filter_by(Plane.XY).sort_by_distance(Vector())[0]
  bottom_face = faces().filter_by(Plane.XY).sort_by(Axis.Z)[0]
  top_inner_surface = faces().filter_by(Plane.XY).sort_by(Axis.Z)[-2]

  with BuildSketch(Plane.XY) as s:
    Circle(radius - outer_wall_thickness)

    with BuildSketch(Plane.XY, mode=Mode.SUBTRACT):
      lines = []
      inners = []
      splitted = [shaft.edges().filter_by(GeomType.LINE)[0].trim(0.1, 0.9).reversed()]
      for i in range(3):
        splitted.append(
          shaft.edges().filter_by(GeomType.CIRCLE)[0].trim(i / 3, (i + 1) / 3).trim(0.1, 0.9)
        )
      for edge in splitted:
        with BuildLine() as l:
          add(edge)
          offset(amount=wall_thickness, side=Side.RIGHT, mode=Mode.REPLACE)
          outer: Edge = edges().group_by(SortBy.LENGTH)[-1].sort_by_distance(Vector())[-1]
          line_dir = (outer.center() - Vector()).normalized()
        inners.append(make_face(mode=Mode.PRIVATE))
        s0 = outer.length * 0.5
        lines.append(
          IntersectingLine(
            outer.position_at(s0 + wall_thickness / 2, position_mode=PositionMode.LENGTH),
            line_dir,
            inner_bottom_edge,
            mode=Mode.PRIVATE,
          )
        )
        lines.append(
          IntersectingLine(
            outer.position_at(s0 - wall_thickness / 2, position_mode=PositionMode.LENGTH),
            line_dir,
            inner_bottom_edge,
            mode=Mode.PRIVATE,
          )
        )
      
      with BuildLine() as l:
        for e1, e2 in zip(lines[1::2], lines[::2]):
          add([e1, e2])
          Line(e1 @ 0, e2 @ 0)

        pairs = [(lines[(i+1) % len(lines)], lines[i]) for i in range(1, len(lines), 2)]
        for e1, e2 in pairs:
          RadiusArc(e1 @ 1, e2 @ 1, radius - outer_wall_thickness)
      make_face()
      for face in inners:
        add(face, mode=Mode.SUBTRACT)

      sharp = vertices()
      fillet(sharp, wall_thickness / 3)
  extrude(s.sketch, amount=height-outer_wall_thickness)


encoder_knob = encoder_knob.part
encoder_knob.label = "encoder_knob"

show_object(encoder_knob)

+


In [249]:
export(encoder_knob)

Exported 'encoder_knob' to export/encoder_knob.v2.step
